In [1]:
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

## Data

### PPI

In [2]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [3]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

100%|██████████| 13715404/13715404 [00:02<00:00, 5296424.93it/s]


Number of pairs missing their reverse: 0
All pairs have their reverse present.


In [4]:
protein_interaction

,protein1,protein2,combined_score,Translated_protein_1,Translated_protein_2
0,9606.ENSP00000000233,9606.ENSP00000356607,173,ARF5,RALGPS2
1,9606.ENSP00000000233,9606.ENSP00000427567,154,ARF5,FHDC1
2,9606.ENSP00000000233,9606.ENSP00000253413,151,ARF5,ATP6V1E1
3,9606.ENSP00000000233,9606.ENSP00000493357,471,ARF5,CYTH2
4,9606.ENSP00000000233,9606.ENSP00000324127,201,ARF5,PSD3
...,...,...,...,...,...
13715399,9606.ENSP00000501317,9606.ENSP00000475489,195,RFX7,MPHOSPH9
13715400,9606.ENSP00000501317,9606.ENSP00000370447,158,RFX7,VCX
13715401,9606.ENSP00000501317,9606.ENSP00000312272,226,RFX7,YPEL2
13715402,9606.ENSP00000501317,9606.ENSP00000402092,169,RFX7,SAMD3


### DrugBank

In [5]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [6]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [7]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

Found 17430 drugs in the DrugBank XML.


### Genetic results

In [8]:
### import data

### genes
hpv_positive_genes  = pd.read_csv('Results/CNV results/HPV positive CNV top genes.csv')
hpv_negative_genes = pd.read_csv('Results/CNV results/HPV negative CNV top genes.csv')

### drug candiates
hpv_positive_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Direct Drug Candidates Aggregated.csv')
hpv_positive_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Indirect Drug Candidates Aggregated.csv')

hpv_negative_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Direct Drug Candidates Aggregated.csv')
#### no direct drug candidates came from Deletions, only amplifications
hpv_negative_direct_drug_candidates['MUT_TYPE'] = 'AMPLIFICATION'
hpv_negative_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Indirect Drug Candidates Aggregated.csv')

### somatic mtuation
hpv_positive_som_genes = pd.read_csv('Results/SOM results/HPV positive top genes.csv')
hpv_positive_som_direct_drug_candidates = pd.read_csv('Results/SOM results/hpv_positive_som_top_direct_drug_candidates_agg.csv')
hpv_positive_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_indirect_drug_candidates = pd.read_csv('Results/SOM results/hpv_positive_som_top_indirect_drug_candidates_agg.csv')
hpv_positive_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

hpv_negative_som_genes = pd.read_csv('Results/SOM results/HPV negative top genes.csv')
hpv_negative_som_direct_drug_candidates = pd.read_csv('Results/SOM results/hpv_negative_som_top_direct_drug_candidates_agg.csv')
hpv_negative_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_indirect_drug_candidates = pd.read_csv('Results/SOM results/hpv_negative_som_top_indirect_drug_candidates_agg.csv')
hpv_negative_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

#### Overlap

##### HPV+

In [9]:
### number of unique drugs of all hpv positive both direct and indirect
num_unique_pos_drugs = len(list(set(list(set(hpv_positive_direct_drug_candidates['DRUG'].str.lower()))
                           + list(set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())))))
num_unique_pos_drugs

168

In [10]:
# Get the set of unique HPV positive direct drug candidates
hpv_positive_direct_drugs = set(hpv_positive_direct_drug_candidates['DRUG'].str.lower())
hpv_positive_som_direct_drugs = set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive direct drug candidates
all_hpv_positive_direct_drugs = hpv_positive_direct_drugs.union(hpv_positive_som_direct_drugs)

print(f"Number of unique HPV positive direct drug candidates: {len(all_hpv_positive_direct_drugs)}")
print("\nHPV positive direct drug candidates:")
print(sorted(all_hpv_positive_direct_drugs))

Number of unique HPV positive direct drug candidates: 14

HPV positive direct drug candidates:
['buparlisib', 'ch-5132799', 'cladribine', 'copanlisib', 'copper', 'golotimod', 'nadh', 'tg-100801', 'wortmannin', 'xl765', 'zinc', 'zinc acetate', 'zinc chloride', 'zinc sulfate, unspecified form']


In [11]:
# Get the set of unique HPV positive indirect drug candidates
hpv_positive_indirect_drugs = set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())
hpv_positive_som_indirect_drugs = set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive indirect drug candidates
all_hpv_positive_indirect_drugs = hpv_positive_indirect_drugs.union(hpv_positive_som_indirect_drugs)

print(f"Number of unique HPV positive indirect drug candidates: {len(all_hpv_positive_indirect_drugs)}")
print("\nHPV positive indirect drug candidates:")
print(sorted(all_hpv_positive_indirect_drugs))

Number of unique HPV positive indirect drug candidates: 163

HPV positive indirect drug candidates:
['1-tert-butyl-3-(4-chloro-phenyl)-1h-pyrazolo[3,4-d]pyrimidin-4-ylamine', '2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imidazo[4,5-f]isoquinolin-7-one', '[4-({5-(aminocarbonyl)-4-[(3-methylphenyl)amino]pyrimidin-2-yl}amino)phenyl]acetic acid', 'abrocitinib', 'aceclidine', 'aclidinium', 'afatinib', 'alteplase', 'altiratinib', 'amuvatinib', 'anisotropine methylbromide', 'aripiprazole', 'aripiprazole lauroxil', 'artenimol', 'axitinib', 'baricitinib', 'benzquinamide', 'bethanechol', 'bimiralisib', 'bisindolylmaleimide i', 'bms-690514', 'bms-754807', 'bosutinib', 'brigatinib', 'brompheniramine', 'buparlisib', 'cabozantinib', 'canertinib', 'capivasertib', 'cerdulatinib', 'ch-5132799', 'chlorprothixene', 'ci-1040', 'cladribine', 'clozapine', 'conestat alfa', 'copanlisib', 'crizotinib', 'dacomitinib', 'darifenacin', 'dasatinib', 'debio-1347', 'delgocitinib', 'deuruxolitinib', 'dihydroergotamine', 

In [12]:
# Get the set of unique HPV positive direct drug candidates
hpv_positive_direct_drugs = set(hpv_positive_direct_drug_candidates['DRUG'].str.lower())
hpv_positive_som_direct_drugs = set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())

# Get the set of unique HPV positive indirect drug candidates
hpv_positive_indirect_drugs = set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())
hpv_positive_som_indirect_drugs = set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive direct and indirect drug candidates
all_hpv_positive_direct_drugs = hpv_positive_direct_drugs.union(hpv_positive_som_direct_drugs)
all_hpv_positive_indirect_drugs = hpv_positive_indirect_drugs.union(hpv_positive_som_indirect_drugs)


# Find the overlap between direct and indirect drug candidates
overlapping_drugs = all_hpv_positive_direct_drugs.intersection(all_hpv_positive_indirect_drugs)

print(f"Number of unique HPV positive direct drug candidates: {len(all_hpv_positive_direct_drugs)}")
print(f"Number of unique HPV positive indirect drug candidates: {len(all_hpv_positive_indirect_drugs)}")
print(f"Number of overlapping drugs between direct and indirect: {len(overlapping_drugs)}")
print("\nOverlapping drugs:")
print(sorted(overlapping_drugs))

Number of unique HPV positive direct drug candidates: 14
Number of unique HPV positive indirect drug candidates: 163
Number of overlapping drugs between direct and indirect: 9

Overlapping drugs:
['buparlisib', 'ch-5132799', 'cladribine', 'copanlisib', 'golotimod', 'nadh', 'tg-100801', 'wortmannin', 'xl765']


##### HPV-

In [13]:
### number of unique drugs of all hpv negative both direct and indirect
num_unique_neg_drugs = len(list(set(list(set(hpv_negative_direct_drug_candidates['DRUG'].str.lower()))
                           + list(set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())))))
num_unique_neg_drugs

110

In [14]:
## HPV- Drug Candidates
### Number of unique direct drug candidates
# Get the set of unique HPV negative direct drug candidates
hpv_negative_direct_drugs = set(hpv_negative_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_som_direct_drugs = set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV negative direct drug candidates
all_hpv_negative_direct_drugs = hpv_negative_direct_drugs.union(hpv_negative_som_direct_drugs)

print(f"Number of unique HPV negative direct drug candidates: {len(all_hpv_negative_direct_drugs)}")
print("\nHPV negative direct drug candidates:")
print(sorted(all_hpv_negative_direct_drugs))


Number of unique HPV negative direct drug candidates: 20

HPV negative direct drug candidates:
['3-isobutyl-1-methyl-7h-xanthine', 'acetylsalicylic acid', 'biotin', 'bisindolylmaleimide i', 'caffeine', 'copper', 'dasatinib', 'ethanol', 'fludiazepam', 'fostamatinib', 'glutamic acid', 'nadh', 'phenethyl isothiocyanate', 'regorafenib', 'wortmannin', 'xl765', 'zinc', 'zinc acetate', 'zinc chloride', 'zinc sulfate, unspecified form']


In [15]:
# Get the set of unique HPV negative indirect drug candidates
hpv_negative_indirect_drugs = set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())
hpv_negative_som_indirect_drugs = set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV negative indirect drug candidates
all_hpv_negative_indirect_drugs = hpv_negative_indirect_drugs.union(hpv_negative_som_indirect_drugs)

print(f"Number of unique HPV negative indirect drug candidates: {len(all_hpv_negative_indirect_drugs)}")
print("\nHPV negative indirect drug candidates:")
print(sorted(all_hpv_negative_indirect_drugs))

Number of unique HPV negative indirect drug candidates: 104

HPV negative indirect drug candidates:
['3-isobutyl-1-methyl-7h-xanthine', 'aceclidine', 'acetylsalicylic acid', 'aclidinium', 'ag-24322', 'alsterpaullone', 'altiratinib', 'alvocidib', 'amuvatinib', 'aripiprazole lauroxil', 'arsenic trioxide', 'artenimol', 'bethanechol', 'biotin', 'bisindolylmaleimide i', 'bms-690514', 'brigatinib', 'caffeine', 'canertinib', 'carfilzomib', 'chlorprothixene', 'cholic acid', 'ci-1040', 'clozapine', 'conestat alfa', 'darifenacin', 'dasatinib', 'enzastaurin', 'erdafitinib', 'ethanol', 'famitinib', 'fesoterodine', 'fg-9041', 'fludiazepam', 'fn-1501', 'foreskin keratinocyte (neonatal)', 'fostamatinib', 'gamma-aminobutyric acid', 'glutamic acid', 'glycopyrronium', 'heparin', 'homatropine', 'homatropine methylbromide', 'human c1-esterase inhibitor', 'hymenialdisine', 'imatinib', 'lenvatinib', 'linifanib', 'lucitanib', 'lutetium lu 177 dotatate', 'martinostat', 'meprobamate', 'methacholine', 'midostau

In [16]:
# Get the set of unique HPV negative drug candidates (both direct and indirect)
hpv_negative_direct_drugs = set(hpv_negative_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_som_direct_drugs = set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_indirect_drugs = set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())
hpv_negative_som_indirect_drugs = set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine all HPV negative drug sets
all_hpv_negative_direct_drugs = hpv_negative_direct_drugs.union(hpv_negative_som_direct_drugs)
all_hpv_negative_direct_drugs = set(drug.lower() for drug in all_hpv_negative_direct_drugs)
all_hpv_negative_indirect_drugs = hpv_negative_indirect_drugs.union(hpv_negative_som_indirect_drugs)
all_hpv_negative_indirect_drugs = set(drug.lower() for drug in all_hpv_negative_indirect_drugs)

# Find the overlap between direct and indirect drug candidates for HPV negative
overlapping_drugs = all_hpv_negative_direct_drugs.intersection(all_hpv_negative_indirect_drugs)


print(f"Number of unique HPV negative direct drug candidates: {len(all_hpv_negative_direct_drugs)}")
print(f"Number of unique HPV negative indirect drug candidates: {len(all_hpv_negative_indirect_drugs)}")
print(f"Number of overlapping drugs between direct and indirect: {len(overlapping_drugs)}")
print("\nOverlapping drugs:")
print(sorted(overlapping_drugs))

Number of unique HPV negative direct drug candidates: 20
Number of unique HPV negative indirect drug candidates: 104
Number of overlapping drugs between direct and indirect: 14

Overlapping drugs:
['3-isobutyl-1-methyl-7h-xanthine', 'acetylsalicylic acid', 'biotin', 'bisindolylmaleimide i', 'caffeine', 'dasatinib', 'ethanol', 'fludiazepam', 'fostamatinib', 'glutamic acid', 'nadh', 'phenethyl isothiocyanate', 'regorafenib', 'wortmannin']


#### Literature results

In [17]:
extracted_target_df= pd.read_csv('Validation pipeline/Results/cleaned_extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv')
extracted_target_df_combined = pd.read_csv('Validation pipeline/Results/cleaned_extracted_combined_targets_all_pub_after_2000_GPU_2b_gemma.csv')

In [18]:
### accumulate all genes available in drugbank or ppi
Drug_bank_genes = list(Drug_bank['gene'].values)
ppi_genes = list(protein_interaction['Translated_protein_1'].values)
all_ppi_drugbank = list(set(Drug_bank_genes + ppi_genes))

## HPV+

#### Genes

In [19]:
hpv_positive_som_genes

,Gene,Count,Cohort_Frequency,Normalized_Count,Normalized_Cohort_Frequency,P_Value,Adjusted_P_Value,Significant,Empirical_P_Value,Adjusted_Empirical_P_Value,frequency_percentage,mutation_score
0,PIK3CA,18,17,0.002182,0.002061,1.055322e-21,4.630755e-18,True,0.0,0.0,23.611111,0.051515
1,ZNF750,11,8,0.003502,0.002547,4.795352e-16,1.052100e-12,True,0.0,0.0,11.111111,0.038912
2,CCDC191,6,6,0.001148,0.001148,1.514273e-06,1.107438e-03,True,0.0,0.0,8.333333,0.009564
3,FAM135B,6,5,0.001229,0.001024,1.024534e-06,1.107438e-03,True,0.0,0.0,6.944444,0.008533
4,AGO4,4,4,0.001549,0.001549,2.791318e-05,1.034515e-02,True,0.0,0.0,5.555556,0.008603
5,PRPF6,4,4,0.001417,0.001417,3.933866e-05,1.327831e-02,True,0.0,0.0,5.555556,0.007872


In [20]:
### combine hpv positive somatic genes and cnv genes
hpv_positive_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_genes['gene_name'] = hpv_positive_som_genes['Gene']
hpv_positive_som_genes['q_value']= hpv_positive_som_genes['Adjusted_P_Value']
hpv_positive_som_genes['empirical_q_value'] = hpv_positive_som_genes['Adjusted_Empirical_P_Value']
hpv_positive_combined_genes = pd.concat([hpv_positive_genes, hpv_positive_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_positive_combined_genes = hpv_positive_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()

hpv_positive_combined_genes


,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ACE2,DELETION,3.07824363509455e-08,0.0254037070312238
1,ACTL6A,AMPLIFICATION,4.9013845356499404e-54,0.0116131232142737
2,ADIPOQ,AMPLIFICATION,4.9013845356499404e-54,0.0116131232142737
3,AGO4,SOMATIC,0.0103451516326857,0.0
4,AHSG,AMPLIFICATION,4.9013845356499404e-54,0.0116131232142737
5,ANOS1,DELETION,3.07824363509455e-08,0.0254037070312238
6,BMX,DELETION,3.07824363509455e-08,0.0254037070312238
7,CCDC191,SOMATIC,0.0011074380273068,0.0
8,CLDN1,AMPLIFICATION,6.5911053664735535e-53,0.0116131232142737
9,CNKSR2,DELETION,3.07824363509455e-08,0.0254037070312238


In [21]:
len(set(hpv_positive_combined_genes['gene_name']))

51

In [22]:
### merge genes with number of articles, pubmed id from literature data
hpv_positive_genes_with_lit = pd.merge(hpv_positive_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_positive_genes_with_lit.drop(columns =['INDEX'], inplace = True)

In [23]:
hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
8,CLDN1,AMPLIFICATION,6.5911053664735535e-53,0.0116131232142737,CLDN1,"15170668, 17091452",2.0
30,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499404e-54, 4.6307549428396854e-18","0.0116131232142737, 0.0",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
36,RFC4,AMPLIFICATION,4.9013845356499404e-54,0.0116131232142737,RFC4,16467079,1.0
41,SOX2,AMPLIFICATION,1.2920437544143296e-55,0.0116131232142737,SOX2,15942670,1.0
43,TLR7,DELETION,3.07824363509455e-08,0.0254037070312238,TLR7,17201162,1.0


In [24]:
hpv_positive_genes_with_lit = hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

In [25]:
hpv_positive_genes_with_lit.to_csv('Results/HPV positive gene results.csv')

In [26]:
### export to final results output
hpv_positive_genes_with_lit.to_csv('Results/Final Results/HPV Positive validated genes.csv')

#### Direct

In [27]:
### merge all hpv postive direct drug candidates
hpv_positive_final_direct = pd.concat([hpv_positive_direct_drug_candidates, hpv_positive_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value	drug_hypergeom_fdr	drug_empirical_p_value	drug_empirical_fdr	MUT_TYPE	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	GENE_SIGNIFICANT
hpv_positive_final_direct = hpv_positive_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x),
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [28]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00060,0.040730
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00060,0.040730
2,Cladribine,POLA1,DELETION,1,12,8.333333,inhibitor,chromatin binding,9.203648e-05,2.973096e-02,0.00007,0.025221
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00060,0.040730
4,Copper,"AHSG, KNG1","AMPLIFICATION, AMPLIFICATION",2,146,1.369863,None,cysteine-type endopeptidase inhibitor activity,8.218045e-15,2.566222e-11,0.00001,0.002179
5,Golotimod,TLR7,DELETION,1,5,20.000000,None,double-stranded RNA binding,1.605536e-05,6.539420e-03,0.00004,0.015613
6,NADH,NDUFB5,AMPLIFICATION,1,144,0.694444,None,NADH dehydrogenase (ubiquinone) activity,6.205177e-08,2.325204e-05,0.00001,0.002179
7,TG-100801,VEGFD,DELETION,1,8,12.500000,inhibitor,chemoattractant activity,4.038151e-05,1.576225e-02,0.00001,0.004461
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.000000,None,1-phosphatidylinositol-3-kinase activity,5.225777e-06,8.159180e-04,0.00001,0.001588
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.000000,None,1-phosphatidylinositol-3-kinase activity,7.766063e-05,8.661009e-03,0.00011,0.011982


In [29]:
extracted_target_df_combined

,GENE,PMID,INDEX,NUMBER_OF_ARTICLES
0,000-2,"11302242, 11302242","2610, 2610",1
1,10,"12608845, 12768769, 14967420, 15193028, 180565...","15288, 17016, 27005, 29481, 57658, 61105",6
2,106PRE,18186293,58737,1
3,106R,18186293,58737,1
4,106RECR,18186293,58737,1
...,...,...,...,...
6072,ZP-V3,12464647,13458,1
6073,ZP-V4,12464647,13458,1
6074,ZYGOMA,15883929,36459,1
6075,ZYGOMATIC,"12775236, 17522494","17081, 53057",2


In [30]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES

hpv_positive_final_direct['PMID'] = ''
hpv_positive_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    hpv_positive_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(list(set(literature_gene_targets)))
    hpv_positive_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

In [31]:
hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00060,0.040730,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00060,0.040730,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00060,0.040730,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006539,0.00004,0.015613,17201162,1,TLR7
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000005,0.000816,0.00001,0.001588,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008661,0.00011,0.011982,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA


In [32]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_positive_final_direct = hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_positive_final_direct.to_csv('Results/HPV Positive direct results.csv')

#### Indirect

In [33]:
### merge all hpv positive indirect drug candidates
hpv_positive_final_indirect = pd.concat([hpv_positive_indirect_drug_candidates, hpv_positive_som_indirect_drug_candidates], ignore_index=True)
hpv_positive_final_indirect['ACTION'] = hpv_positive_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['SPECIFIC_FUNCTION'] = hpv_positive_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_hypergeom_fdr'] = hpv_positive_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_empirical_fdr'] = hpv_positive_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['MUT_TYPE'] = hpv_positive_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### aggregate/group by drug name
### columns: DRUG	CONNECTED_TO (risk gene)	
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank	
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene	
# GENE_TARGET	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr

hpv_positive_final_indirect = hpv_positive_final_indirect.groupby(['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()
hpv_positive_final_indirect.sort_values(by = 'drug_empirical_fdr', ascending = True).head(25)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
58,fruquintinib,"KDR, FLT1, FLT4",PIK3CA,3,3,100.000000,inhibitor,ATP binding,3.802145e-03,0.001588,SOMATIC
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001588,SOMATIC
72,imc-1c11,"KDR, FLT1, FLT4",PIK3CA,3,3,100.000000,UNKNOWN,ATP binding,3.802145e-03,0.001588,SOMATIC
159,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,4.977113e-07,0.001588,SOMATIC
158,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,4,4,100.000000,UNKNOWN,ATP binding,1.931716e-04,0.001588,SOMATIC
157,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.220718e-05,0.001588,SOMATIC
154,wortmannin,"PIK3CD, PIK3CG, PIK3R1",PIK3CA,3,5,60.000000,inhibitor,"1-phosphatidylinositol-3-kinase activity, 1-ph...",8.159180e-04,0.001588,SOMATIC
74,"inositol 1,3,4,5-tetrakisphosphate","PDPK1, BTK, CYTH2, CYTH3, AKT1",PIK3CA,5,8,62.500000,inhibitor,3-phosphoinositide-dependent protein kinase ac...,3.097531e-04,0.001588,SOMATIC
8,altiratinib,"MET, KDR, NTRK1, NTRK3",PIK3CA,4,5,80.000000,"inhibitor, antagonist",ATP binding,8.159180e-04,0.001588,SOMATIC
22,bosutinib,"HCK, FGR, SRC, MAP2K1, ABL1, LYN",PIK3CA,6,11,54.545455,inhibitor,"ATP binding, actin filament binding",1.295998e-04,0.001588,SOMATIC


In [34]:
hpv_positive_final_indirect[hpv_positive_final_indirect['ACTION']!= 'UNKNOWN'].sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending = [ True, False ]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
58,fruquintinib,"KDR, FLT1, FLT4",PIK3CA,3,3,100.000000,inhibitor,ATP binding,3.802145e-03,0.001588,SOMATIC
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001588,SOMATIC
8,altiratinib,"MET, KDR, NTRK1, NTRK3",PIK3CA,4,5,80.000000,"inhibitor, antagonist",ATP binding,8.159180e-04,0.001588,SOMATIC
121,ruxolitinib,"JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001588,SOMATIC
49,entrectinib,"ROS1, JAK2, ALK, NTRK1, NTRK3",PIK3CA,5,7,71.428571,inhibitor,"ATP binding, acetylcholine receptor binding",1.915567e-04,0.001588,SOMATIC
74,"inositol 1,3,4,5-tetrakisphosphate","PDPK1, BTK, CYTH2, CYTH3, AKT1",PIK3CA,5,8,62.500000,inhibitor,3-phosphoinositide-dependent protein kinase ac...,3.097531e-04,0.001588,SOMATIC
154,wortmannin,"PIK3CD, PIK3CG, PIK3R1",PIK3CA,3,5,60.000000,inhibitor,"1-phosphatidylinositol-3-kinase activity, 1-ph...",8.159180e-04,0.001588,SOMATIC
22,bosutinib,"HCK, FGR, SRC, MAP2K1, ABL1, LYN",PIK3CA,6,11,54.545455,inhibitor,"ATP binding, actin filament binding",1.295998e-04,0.001588,SOMATIC
23,brigatinib,"IGF1R, MET, ALK, ERBB4, INSR, EGFR, ABL1, ERBB...","PIK3CA,VEGFD, PIK3CA",9,9,100.000000,"inhibitor, binding, inhibitor, binding","ATP binding, amyloid-beta binding, actin filam...",1.590798e-05,0.002179,"DELETION,AMPLIFICATION, SOMATIC"
35,conestat alfa,"C1R, C1S,PLAT, F12, F2,KLKB1, F11","MASP1,HRG,KNG1",7,7,100.000000,inhibitor,calcium ion binding,4.544467e-04,0.002179,AMPLIFICATION


In [35]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_indirect_genes = extract_unique_gene_targets(hpv_positive_indirect_drug_candidates, 'GENE_TARGET')
print(hpv_pos_indirect_genes)

263


In [ ]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES
hpv_positive_final_indirect['PMID'] = ''
hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(',')
    gene_targets = [gene.strip() for gene in gene_targets]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

### validate risk genes
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect['RISK_GENE_PMID'] = ''
hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        #print(gene)
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### ensure that only drugs with NUMBER_OF_ARTICLES > 0 for both drug targets and risk genes are saved, so that they have literature support
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

In [90]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_final_indirect_genes = extract_unique_gene_targets(hpv_positive_final_indirect, 'GENE_TARGET')
print(hpv_pos_final_indirect_genes)


156


In [95]:
hpv_positive_final_indirect[hpv_positive_final_indirect['DRUG']=='artenimol']

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS,Target_Description
13,artenimol,"GAPDH,NPM1,ANXA2,CCT3, HSPA8,RPS13, RPS6, EEF1...","SOX2,PIK3CA,AHSG,DNAJB11,EIF4A2,FXR1,HRG,RPL39...",36,104,34.615385,ligand,aspartic-type endopeptidase inhibitor activity,0.000068,0.002179,"DELETION,AMPLIFICATION","17132224, 17565818, 17663946, 16761435, 164552...",15,"RPL14, RPS13, EEF1A1, ANXA2, RPS6, HSPA8, GAPDH","18447972, 11836556, 11959846, 16815198, 15...",18,"PIK3CA, SOX2",Indirect


In [96]:
hpv_positive_final_indirect.to_csv('Results/HPV Positive indirect results.csv', index=False)

In [97]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_final_indirect_genes = extract_unique_gene_targets(hpv_positive_final_indirect, 'GENE_TARGET')
print(hpv_pos_final_indirect_genes)

156


In [98]:
hpv_positive_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS,Target_Description
0,1-tert-butyl-3-(4-chloro-phenyl)-1h-pyrazolo[3...,"SRC, LCK, LYN",PIK3CA,3,3,100.000000,UNKNOWN,ATP binding,3.802145e-03,0.004866,SOMATIC,"17961551, 17252232",2,"SRC, LYN","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA,Indirect
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001588,SOMATIC,"18204781, 15947106",2,"JAK3, JAK2","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA,Indirect
3,abrocitinib,"JAK2, TYK2, JAK3, JAK1, JAK2, TYK2, JAK3, JAK1","PIK3CA, PIK3CA",4,4,100.000000,"inhibitor, inhibitor","acetylcholine receptor binding, ATP binding, a...",3.433760e-02,0.040366,"AMPLIFICATION, SOMATIC","18204781, 15947106",4,"JAK3, JAK2","18447972, 11836556, 11959846, 16815198, 1...",34,PIK3CA,Indirect
6,afatinib,"ERBB4, EGFR, ERBB2",PIK3CA,3,3,100.000000,inhibitor,"ATP binding, actin filament binding",3.802145e-03,0.004866,SOMATIC,"17888033, 12142091, 16144930, 18383530, 158161...",325,"ERBB2, EGFR, ERBB4","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA,Indirect
8,altiratinib,"MET, KDR, NTRK1, NTRK3",PIK3CA,4,5,80.000000,"inhibitor, antagonist",ATP binding,8.159180e-04,0.001588,SOMATIC,"15735049, 11279608, 14991901, 11705871, 153197...",15,"NTRK1, MET, NTRK3","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA,Indirect
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,xl228,"IGF1R, SRC, ABL1",PIK3CA,3,4,75.000000,UNKNOWN,"ATP binding, actin filament binding",1.400202e-02,0.015968,SOMATIC,"15221937, 17961551",2,"SRC, IGF1R","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA,Indirect
157,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.220718e-05,0.001588,SOMATIC,"15355912, 15833854, 16927414, 17047074, 173420...",8,MTOR,"18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA,Indirect
158,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,4,4,100.000000,UNKNOWN,ATP binding,1.931716e-04,0.001588,SOMATIC,"17317803, 17935283, 17513510, 16630292, 144996...",9,"KIT, PDGFRA","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA,Indirect
159,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,4.977113e-07,0.001588,SOMATIC,"16350727, 18059337, 15719371, 17471236, 129076...",55,"RET, FGFR3, FGFR1","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA,Indirect


In [99]:
hpv_positive_final_indirect[hpv_positive_final_indirect['DRUG']=='artenimol']

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS,Target_Description
13,artenimol,"GAPDH,NPM1,ANXA2,CCT3, HSPA8,RPS13, RPS6, EEF1...","SOX2,PIK3CA,AHSG,DNAJB11,EIF4A2,FXR1,HRG,RPL39...",36,104,34.615385,ligand,aspartic-type endopeptidase inhibitor activity,0.000068,0.002179,"DELETION,AMPLIFICATION","17132224, 17565818, 17663946, 16761435, 164552...",15,"RPL14, RPS13, EEF1A1, ANXA2, RPS6, HSPA8, GAPDH","18447972, 11836556, 11959846, 16815198, 15...",18,"PIK3CA, SOX2",Indirect


In [43]:
hpv_positive_final_indirect.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending=[True, False]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
158,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,4,4,100.000000,UNKNOWN,ATP binding,1.931716e-04,0.001588,SOMATIC,"17317803, 17935283, 17513510, 16630292, 144996...",9,"KIT, PDGFRA","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA
159,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,4.977113e-07,0.001588,SOMATIC,"16350727, 18059337, 15719371, 17471236, 129076...",55,"RET, FGFR3, FGFR1","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001588,SOMATIC,"18204781, 15947106",2,"JAK3, JAK2","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA
8,altiratinib,"MET, KDR, NTRK1, NTRK3",PIK3CA,4,5,80.000000,"inhibitor, antagonist",ATP binding,8.159180e-04,0.001588,SOMATIC,"15735049, 11279608, 14991901, 11705871, 153197...",15,"NTRK1, MET, NTRK3","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA
121,ruxolitinib,"JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001588,SOMATIC,"18204781, 15947106",2,"JAK3, JAK2","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA
157,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.220718e-05,0.001588,SOMATIC,"15355912, 15833854, 16927414, 17047074, 173420...",8,MTOR,"18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA
49,entrectinib,"ROS1, JAK2, ALK, NTRK1, NTRK3",PIK3CA,5,7,71.428571,inhibitor,"ATP binding, acetylcholine receptor binding",1.915567e-04,0.001588,SOMATIC,"11279608, 11485898, 15214912, 11216646, 164836...",10,"NTRK1, ALK, NTRK3, JAK2","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA
74,"inositol 1,3,4,5-tetrakisphosphate","PDPK1, BTK, CYTH2, CYTH3, AKT1",PIK3CA,5,8,62.500000,inhibitor,3-phosphoinositide-dependent protein kinase ac...,3.097531e-04,0.001588,SOMATIC,"15896313, 16778075, 17974918, 12532415, 145813...",6,AKT1,"18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA
22,bosutinib,"HCK, FGR, SRC, MAP2K1, ABL1, LYN",PIK3CA,6,11,54.545455,inhibitor,"ATP binding, actin filament binding",1.295998e-04,0.001588,SOMATIC,"16224162, 17961551, 17252232",3,"SRC, HCK, LYN","18447972, 11836556, 11959846, 16815198, 1...",17,PIK3CA
23,brigatinib,"IGF1R, MET, ALK, ERBB4, INSR, EGFR, ABL1, ERBB...","PIK3CA,VEGFD, PIK3CA",9,9,100.000000,"inhibitor, binding, inhibitor, binding","ATP binding, amyloid-beta binding, actin filam...",1.590798e-05,0.002179,"DELETION,AMPLIFICATION, SOMATIC","17888033, 12142091, 16144930, 18383530, 158161...",725,"ALK, IGF1R, MET, ERBB4, ERBB2, EGFR","18447972, 11836556, 11959846, 16815198, 1...",34,PIK3CA


#### overall

In [44]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00060,0.040730,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00060,0.040730,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00060,0.040730,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006539,0.00004,0.015613,17201162,1,TLR7
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000005,0.000816,0.00001,0.001588,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008661,0.00011,0.011982,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA


In [45]:
'artenimol' in hpv_positive_final_indirect['DRUG'].values

True

In [46]:
### combined direct and indirect final results
### final columns: DRUG   GENE_TARGET CONNECTED_TO (risk gene)	NUM_DIRECT_TARGETS_HIT  
# Number of risk or immediate neighbor genes targeted	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr	
### column for Target description: direct, or indirect
hpv_positive_final_direct['Target_Description'] = 'Direct'
hpv_positive_final_indirect['Target_Description'] = 'Indirect'
hpv_positive_final_results = pd.concat([hpv_positive_final_direct, hpv_positive_final_indirect], ignore_index=True)
### replace any null with 'NA' in the hwole dataframe
hpv_positive_final_results = hpv_positive_final_results.fillna('NA')
hpv_positive_final_results = hpv_positive_final_results.groupby('DRUG').agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'NUM_DIRECT_TARGETS_HIT': 'first',
    'Number of risk or immediate neighbor genes targeted': 'first',
    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'max',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'Target_Description': lambda x: ', '.join(x)
}).reset_index()

/var/folders/5p/swntgnbj3fbfxkx02kt3fq980000gn/T/ipykernel_19839/4020300996.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hpv_positive_final_direct['Target_Description'] = 'Direct'


In [47]:
hpv_positive_final_results[hpv_positive_final_results['DRUG'] == 'artenimol']

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),NUM_DIRECT_TARGETS_HIT,Number of risk or immediate neighbor genes targeted,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,Target_Description
12,artenimol,"GAPDH,NPM1,ANXA2,CCT3, HSPA8,RPS13, RPS6, EEF1...","SOX2,PIK3CA,AHSG,DNAJB11,EIF4A2,FXR1,HRG,RPL39...",NA,36.0,NA,34.615385,ligand,aspartic-type endopeptidase inhibitor activity,0.000068,0.002179,Indirect


In [48]:
hpv_positive_final_results.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending = [ True, False ]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),NUM_DIRECT_TARGETS_HIT,Number of risk or immediate neighbor genes targeted,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,Target_Description
88,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,NA,4.0,NA,100.000000,UNKNOWN,ATP binding,1.931716e-04,0.001588,Indirect
89,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,NA,6.0,NA,100.000000,UNKNOWN,ATP binding,4.977113e-07,0.001588,Indirect
1,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,NA,4.0,NA,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001588,Indirect
10,altiratinib,"MET, KDR, NTRK1, NTRK3",PIK3CA,NA,4.0,NA,80.000000,"inhibitor, antagonist",ATP binding,8.159180e-04,0.001588,Indirect
72,ruxolitinib,"JAK2, TYK2, JAK3, JAK1",PIK3CA,NA,4.0,NA,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",8.159180e-04,0.001588,Indirect
87,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,NA,4.0,NA,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.220718e-05,0.001588,Indirect
30,entrectinib,"ROS1, JAK2, ALK, NTRK1, NTRK3",PIK3CA,NA,5.0,NA,71.428571,inhibitor,"ATP binding, acetylcholine receptor binding",1.915567e-04,0.001588,Indirect
42,"inositol 1,3,4,5-tetrakisphosphate","PDPK1, BTK, CYTH2, CYTH3, AKT1",PIK3CA,NA,5.0,NA,62.500000,inhibitor,3-phosphoinositide-dependent protein kinase ac...,3.097531e-04,0.001588,Indirect
18,bosutinib,"HCK, FGR, SRC, MAP2K1, ABL1, LYN",PIK3CA,NA,6.0,NA,54.545455,inhibitor,"ATP binding, actin filament binding",1.295998e-04,0.001588,Indirect
6,Wortmannin,PIK3CA,NA,1.0,NA,5.0,20.000000,NA,1-phosphatidylinositol-3-kinase activity,8.159180e-04,0.001588,Direct


In [49]:
len(set(hpv_positive_final_results['DRUG']))

91

## HPV-

#### Genes

In [50]:
### combine hpv negative somatic genes and cnv genes
hpv_negative_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_genes['gene_name'] = hpv_negative_som_genes['Gene']
hpv_negative_som_genes['q_value']= hpv_negative_som_genes['Adjusted_P_Value']
hpv_negative_som_genes['empirical_q_value'] = hpv_negative_som_genes['Adjusted_Empirical_P_Value']
hpv_negative_combined_genes = pd.concat([hpv_negative_genes, hpv_negative_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_negative_combined_genes = hpv_negative_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()
# hpv_negative_combined_genes.sort_values(by='q_value')

In [51]:
hpv_negative_combined_genes

,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ACTL6A,AMPLIFICATION,0.0,0.0038533077280929
1,ADAMTS3,SOMATIC,2.3697738415135614e-07,0.0
2,ADCY2,SOMATIC,3.5624299726954185e-13,0.0
3,ADIPOQ,AMPLIFICATION,0.0,0.0038533077280929
4,AGMO,SOMATIC,0.0001416648596923,0.0
...,...,...,...,...
219,ZNF750,SOMATIC,0.0002549991889244,0.0
220,ZNF804A,SOMATIC,1.20442355265136e-12,0.0
221,ZNF804B,SOMATIC,2.434404690013089e-12,0.0
222,ZNF835,SOMATIC,2.467457130946929e-13,0.0


In [52]:
### merge genes with number of articles, pubmed id from literature data
hpv_negative_gene_results_with_lit = pd.merge(hpv_negative_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_negative_gene_results_with_lit.drop(columns = ['INDEX'], inplace = True)

In [53]:
hpv_negative_gene_results_with_lit = hpv_negative_gene_results_with_lit[hpv_negative_gene_results_with_lit['NUMBER_OF_ARTICLES']>0]

In [54]:
hpv_negative_gene_results_with_lit.sort_values(by = ['empirical_q_value', 'q_value'], ascending=[True, True], inplace=True)

In [55]:
hpv_negative_gene_results_with_lit.drop(columns=['INDEX', 'GENE'], errors='ignore', inplace=True)

In [56]:
hpv_negative_gene_results_with_lit.to_csv('Results/HPV negative gene results.csv')

In [57]:
hpv_negative_gene_results_with_lit

,gene_name,MUT_TYPE,q_value,empirical_q_value,PMID,NUMBER_OF_ARTICLES
204,TP53,SOMATIC,0.0,0.0,"11390535, 11445847, 11445859, 11916556, 125898...",29.0
162,PDE3A,SOMATIC,1.2533369131876885e-07,0.0,15078486,1.0
108,MAGEC1,SOMATIC,1.496570263607284e-13,0.0,16929165,1.0
53,EPHA2,SOMATIC,1.561562265029652e-16,0.0,"12494475, 16309192, 18030354, 18425361, 18485799",5.0
32,COL1A2,SOMATIC,2.018380908180692e-07,0.0,18254958,1.0
81,HRAS,SOMATIC,2.2137739229868114e-21,0.0,16676365,1.0
177,REG1A,SOMATIC,3.5624299726954185e-13,0.0,12901795,1.0
105,LRP1B,SOMATIC,4.228839135032655e-46,0.0,"15172977, 16857411, 16918994",3.0
175,RAC1,SOMATIC,4.606038103918665e-05,0.0,"17234718, 17592548",2.0
10,BCL6,AMPLIFICATION,0.0,0.0038533077280929,"11224600, 11420458, 14685876, 17429099",4.0


In [58]:
len(set(hpv_negative_gene_results_with_lit['gene_name']))

20

In [59]:
### export top genes to ouput final tables
hpv_negative_gene_results_with_lit.to_csv('Results/Final Results/HPV Negative validated genes.csv')

#### Direct

In [60]:
### combine all hpv negative direct drug candidates
hpv_negative_final_direct = pd.concat([hpv_negative_direct_drug_candidates, hpv_negative_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value      

hpv_negative_final_direct = hpv_negative_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x.unique()), ### unique mutation types per drug
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [100]:
### validate hpv negative direct drug candidates with literature data
hpv_negative_final_direct['PMID'] = ''
hpv_negative_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = [gene.strip() for gene in gene_targets]
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

    

In [101]:
hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0].sort_values(by='NUMBER_OF_ARTICLES', ascending=False)

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,Target_Description
1,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,6.224881e-07,0.000486,0.00001,0.004684,"17312781, 12702551, 16012721, 15240783, 129359...",29,TP53,Direct
4,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,2.553685e-04,0.047846,0.00019,0.039553,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA,Direct
14,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00041,0.048455,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA,Direct
15,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00043,0.048455,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA,Direct
9,Fostamatinib,"MAP3K13, PRKCI, TNIK, EPHA2","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,1.720246e-09,0.000002,0.00001,0.002465,"16309192, 18425361, 17990328, 18485799, 180303...",6,"PRKCI, EPHA2",Direct
6,Dasatinib,EPHA2,SOMATIC,1.0,23.0,4.347826,antagonist,ATP binding,1.350788e-06,0.000744,0.00002,0.006691,"16309192, 18425361, 18485799, 18030354, 12494475",5,EPHA2,Direct
13,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,5.157973e-08,0.000104,0.00001,0.004684,"16309192, 18425361, 18485799, 18030354, 12494475",5,EPHA2,Direct
0,3-isobutyl-1-methyl-7H-xanthine,PDE3A,SOMATIC,1.0,15.0,6.666667,inhibitor,"3',5'-cGMP-inhibited cyclic-nucleotide phospho...",2.553685e-04,0.047846,0.00020,0.040730,15078486,1,PDE3A,Direct
3,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.278605e-07,0.000080,0.00001,0.002465,17990328,1,PRKCI,Direct


In [102]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_negative_final_direct= hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_negative_final_direct.to_csv('Results/HPV Negative direct results.csv')

In [103]:
hpv_negative_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,Target_Description
0,3-isobutyl-1-methyl-7H-xanthine,PDE3A,SOMATIC,1.0,15.0,6.666667,inhibitor,"3',5'-cGMP-inhibited cyclic-nucleotide phospho...",2.553685e-04,0.047846,0.00020,0.040730,15078486,1,PDE3A,Direct
1,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,6.224881e-07,0.000486,0.00001,0.004684,"17312781, 12702551, 16012721, 15240783, 129359...",29,TP53,Direct
3,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.278605e-07,0.000080,0.00001,0.002465,17990328,1,PRKCI,Direct
4,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,2.553685e-04,0.047846,0.00019,0.039553,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA,Direct
6,Dasatinib,EPHA2,SOMATIC,1.0,23.0,4.347826,antagonist,ATP binding,1.350788e-06,0.000744,0.00002,0.006691,"16309192, 18425361, 18485799, 18030354, 12494475",5,EPHA2,Direct
9,Fostamatinib,"MAP3K13, PRKCI, TNIK, EPHA2","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,1.720246e-09,0.000002,0.00001,0.002465,"16309192, 18425361, 17990328, 18485799, 180303...",6,"PRKCI, EPHA2",Direct
13,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,5.157973e-08,0.000104,0.00001,0.004684,"16309192, 18425361, 18485799, 18030354, 12494475",5,EPHA2,Direct
14,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00041,0.048455,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA,Direct
15,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00043,0.048455,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA,Direct


#### Indirect

In [104]:
### combine all hpv negative indirect drug candidates
hpv_negative_final_indirect = pd.concat([hpv_negative_indirect_drug_candidates, hpv_negative_som_indirect_drug_candidates])
hpv_negative_final_indirect['ACTION'] = hpv_negative_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['SPECIFIC_FUNCTION'] = hpv_negative_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_hypergeom_fdr'] = hpv_negative_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_empirical_fdr'] = hpv_negative_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['MUT_TYPE'] = hpv_negative_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### group by drug and comma seperate genes and mutation type
### columns: DRUG	CONNECTED_TO (risk gene)
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank  
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene  
# GENE_TARGET	
# GENE_Cohort_Frequency   
# GENE_Normalized_Count	
# GENE_Normalized_Cohort_Frequency

hpv_negative_final_indirect = hpv_negative_final_indirect.groupby (['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()

In [105]:
'artenimol' in hpv_negative_final_indirect['DRUG']

False

In [106]:
hpv_negative_final_indirect[hpv_negative_final_indirect['DRUG']=='xl765']

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE


In [107]:
hpv_negative_final_indirect.sort_values(by = 'drug_empirical_fdr', ascending = True).head(20)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
75,quercetin,"HCK, PIK3CG, ESR1, STAT3, HSP90AA1,LPCAT1, LPC...","PIK3CA,PLD1,ADIPOQ",19,32,59.375000,inhibitor,"ATP binding, 1-phosphatidylinositol-3-kinase a...",9.146549e-04,0.002465,AMPLIFICATION
85,semaxanib,"FGFR1, KDR, PDGFRB, FLT1, KIT, RET, PDGFRA,MPL","PIK3CA,THPO",8,9,88.888889,inhibitor,ATP binding,6.128792e-03,0.002465,AMPLIFICATION
13,biotin,"ACACA, HLCS, PC,ACACB, MCCC1, PCCB, PCCA, MCCC2","MCCC1,EHHADH",8,8,100.000000,"cofactor, substrate","acetyl-CoA carboxylase activity, ATP binding",1.159902e-03,0.002465,AMPLIFICATION
21,cholic acid,"PLA2G1B,CES1","PLD1,NCEH1",16,22,72.727273,inhibitor,bile acid binding,1.355790e-04,0.002465,AMPLIFICATION
87,somatostatin,"OPRM1,SSTR4, SSTR3, SSTR5, SSTR1, SSTR2,OPRD1","GNB4,SST,KNG1",7,7,100.000000,inhibitor,beta-endorphin receptor activity,4.425648e-03,0.002465,AMPLIFICATION
55,nadh,"NDUFA8, NDUFS5, NDUFS2, NDUFA2, NDUFB10, NDUFV...",NDUFB5,58,144,40.277778,binder,"NADH dehydrogenase (ubiquinone) activity, 4 ir...",3.648098e-05,0.002465,AMPLIFICATION
24,conestat alfa,"C1R, C1S,KLKB1, F11,PLAT, F12, F2","MASP1,KNG1,HRG",7,7,100.000000,inhibitor,calcium ion binding,4.425648e-03,0.002465,AMPLIFICATION
43,human c1-esterase inhibitor,"C1R, C1S,KLKB1, F11,PLAT, F12, F2","MASP1,KNG1,HRG",7,7,100.000000,inhibitor,calcium ion binding,4.425648e-03,0.002465,AMPLIFICATION
92,tetrabromo-2-benzotriazole,"MAP2K1, LCK, RPS6KB1, AKT1,PRKCA, MAPK11, MAPK...","PIK3CA,GNB4,DVL3,RFC4,MECOM,CDKN2A, TP53, PIK3...",13,18,72.222222,"inhibitor, inhibitor, inhibitor, inhibitor","ATP binding, 14-3-3 protein binding, [hydroxym...",1.379657e-03,0.004684,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC, SOMATIC"
38,glutamic acid,"SLC7A11, GLS2, GLUD2, GLUD1, ALDH18A1, GPT2, G...","TP53, PDHA2, HTR5A, GLUD2, SLC17A6, GRM3, WARS...",35,61,57.377049,"UNKNOWN, substrate, UNKNOWN, substrate, UNKNOW...","cystine, glutaminase activity, ADP binding, AT...",2.751901e-07,0.004684,"SOMATIC, SOMATIC, SOMATIC, SOMATIC, SOMATIC, S..."


In [108]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_negative_indirect_genes = extract_unique_gene_targets(hpv_negative_final_indirect, 'GENE_TARGET')
print(hpv_negative_indirect_genes)

485


In [109]:
hpv_negative_final_indirect[hpv_negative_final_indirect['DRUG'] == 'artenimol']

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
11,artenimol,"HNRNPK, DDX5,NPM1,ACTG1,ATP5F1A, ATP5MG, ATP5P...","FXR1,PIK3CA,ACTL6A,NDUFB5,SOX2,TNFSF10,P3H2,EI...",40,104,38.461538,"ligand, ligand, ligand, ligand, ligand, ligand...","cadherin binding, ATP binding, cadherin bindin...",0.014303,0.024175,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC, SOMA..."


In [110]:
### add in literature validation columns
hpv_negative_final_indirect['PMID'] = ''
hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(',')
    gene_targets = [gene.strip() for gene in gene_targets]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles


### validate risk genes
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect['RISK_GENE_PMID'] = ''
hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### make sure only drugs with literature support for both drug targets and risk genes are saved
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

hpv_negative_final_indirect.to_csv('Results/HPV Negative indirect results.csv', index=False)



In [111]:
hpv_negative_final_indirect.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending=[True, False]).head(50)[['DRUG', 'drug_empirical_fdr','LITERATURE_GENE_TARGETS', 'RISK_GENE_PMID','RISK_GENE_LITERATURE_GENE_TARGETS','RISK_GENE_NUMBER_OF_ARTICLES','MUT_TYPE']]

,DRUG,drug_empirical_fdr,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_LITERATURE_GENE_TARGETS,RISK_GENE_NUMBER_OF_ARTICLES,MUT_TYPE
85,semaxanib,0.002465,"RET, KIT, FGFR1, PDGFRA","16676365, 17848307, 15543611, 15700036, 179891...",PIK3CA,17,AMPLIFICATION
75,quercetin,0.002465,"ESR1, STAT3, HCK, SHBG","16676365, 17848307, 15543611, 15700036, 179891...",PIK3CA,17,AMPLIFICATION
16,brigatinib,0.004684,"ALK, IGF1R, MET, ERBB4, ERBB2, EGFR","11309301, 11358835, 14581353, 17117177, 126844...","PIK3CA, CDKN2A",50,"DELETION,AMPLIFICATION, SOMATIC"
28,erdafitinib,0.004684,"PDGFRA, FGFR4, FGFR3, KIT, RET, FGFR1, FGFR2","16676365, 17848307, 15543611, 15700036, 179891...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
30,famitinib,0.004684,"KIT, PDGFRA","16676365, 17848307, 15543611, 15700036, 179891...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
46,lenvatinib,0.004684,"PDGFRA, FGFR4, FGFR3, KIT, RET, FGFR1, FGFR2","16676365, 17848307, 15543611, 15700036, 179891...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
58,nintedanib,0.004684,"PDGFRA, SRC, LYN, FGFR3, FGFR1, FGFR2","16676365, 17848307, 15543611, 15700036, 179891...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
62,pd-166326,0.004684,"PDGFRA, SRC, KIT, FGFR1, EGFR","17312781, 12702551, 11309301, 16012721, 113588...","PIK3CA, TP53, CDKN2A",79,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC"
88,sorafenib,0.004684,"KIT, RET, FGFR1, BRAF, EGFR","16676365, 17848307, 15543611, 15700036, 179891...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
91,sunitinib,0.004684,"MET, PDGFRA, KIT","16676365, 17848307, 15543611, 15700036, 179891...",PIK3CA,34,"AMPLIFICATION, SOMATIC"


In [112]:
hpv_negative_final_indirect[hpv_negative_final_indirect['DRUG'] == 'artenimol']

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
11,artenimol,"HNRNPK, DDX5,NPM1,ACTG1,ATP5F1A, ATP5MG, ATP5P...","FXR1,PIK3CA,ACTL6A,NDUFB5,SOX2,TNFSF10,P3H2,EI...",40,104,38.461538,"ligand, ligand, ligand, ligand, ligand, ligand...","cadherin binding, ATP binding, cadherin bindin...",0.014303,0.024175,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC, SOMA...","17132224, 17884789, 17565818, 17663946, 167614...",24,"VIM, LGALS1, RPL14, PRDX1, RPS13, TPI1, RPS6, ...","17312781, 12702551, 11309301, 17234718, 160127...",85,"HRAS, CDKN2A, TP53, SOX2, COL1A2, EIF4G1, PIK3..."


#### overall

In [113]:
hpv_negative_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,Target_Description
0,3-isobutyl-1-methyl-7H-xanthine,PDE3A,SOMATIC,1.0,15.0,6.666667,inhibitor,"3',5'-cGMP-inhibited cyclic-nucleotide phospho...",2.553685e-04,0.047846,0.00020,0.040730,15078486,1,PDE3A,Direct
1,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,6.224881e-07,0.000486,0.00001,0.004684,"17312781, 12702551, 16012721, 15240783, 129359...",29,TP53,Direct
3,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.278605e-07,0.000080,0.00001,0.002465,17990328,1,PRKCI,Direct
4,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,2.553685e-04,0.047846,0.00019,0.039553,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA,Direct
6,Dasatinib,EPHA2,SOMATIC,1.0,23.0,4.347826,antagonist,ATP binding,1.350788e-06,0.000744,0.00002,0.006691,"16309192, 18425361, 18485799, 18030354, 12494475",5,EPHA2,Direct
9,Fostamatinib,"MAP3K13, PRKCI, TNIK, EPHA2","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,1.720246e-09,0.000002,0.00001,0.002465,"16309192, 18425361, 17990328, 18485799, 180303...",6,"PRKCI, EPHA2",Direct
13,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,5.157973e-08,0.000104,0.00001,0.004684,"16309192, 18425361, 18485799, 18030354, 12494475",5,EPHA2,Direct
14,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00041,0.048455,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA,Direct
15,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.396073e-04,0.046272,0.00043,0.048455,"16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA,Direct


In [114]:
hpv_negative_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
2,acetylsalicylic acid,"PRKAA1, PCNA, CASP3, CASP1, IKBKB, NFKBIA, PTG...","TP53, HRAS, PIK3CA, HAS2, UGT2B4",15,19,78.947368,"activator, downregulator, inhibitor, inhibitor...",[hydroxymethylglutaryl-CoA reductase (NADPH)] ...,0.000486,0.004684,"SOMATIC, SOMATIC, SOMATIC, SOMATIC, SOMATIC","11859213, 15846092, 16954163, 14697637, 167572...",114,"MYC, CCND1, TP53, AKR1C1, NFKBIA, PCNA, TNFAIP...","17312781, 12702551, 16012721, 11358835, 145813...",47,"PIK3CA, TP53, HRAS"
4,ag-24322,"CDK1,CDK2, CDK4","CDKN2A,CDKN2B",3,3,100.000000,inhibitor,ATP binding,0.010932,0.007807,DELETION,"12429646, 12212108, 11585773, 16027075, 164964...",29,"CDK2, CDK4, CDK1","16713672, 11309301, 15495191, 17673925, 145866...",18,"CDKN2B, CDKN2A"
5,alsterpaullone,"CDK1, CDK5,CDK2","CDKN2A,CDKN2B",3,4,75.000000,inhibitor,"ATP binding, acetylcholine receptor activator ...",0.034197,0.022483,DELETION,"12017338, 15833870, 16248248, 11453659, 112720...",16,"CDK2, CDK5, CDK1","16713672, 11309301, 15495191, 17673925, 145866...",18,"CDKN2B, CDKN2A"
6,altiratinib,"MET, KDR, NTRK1, NTRK3,TEK","PIK3CA,THPO",5,5,100.000000,"inhibitor, antagonist",ATP binding,0.046272,0.046840,AMPLIFICATION,"15735049, 11279608, 14991901, 11705871, 153197...",15,"NTRK1, MET, NTRK3","16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA
7,alvocidib,"CDK2, CDK4, CDK6, CDK7, CDK6, CDK9, CDK5, CDK8...","CDKN2B, TP53, PIK3CA",6,12,50.000000,"inhibitor, inhibitor, UNKNOWN","ATP binding, ATP binding, 7SK snRNA binding, a...",0.030578,0.039214,"DELETION, SOMATIC, SOMATIC","17888033, 12142091, 16144930, 18383530, 158161...",366,"CDK4, CDK5, CDK2, CDK1, EGFR, CDK6","17312781, 12702551, 17673925, 16012721, 113588...",48,"PIK3CA, TP53, CDKN2B"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,tetrabromo-2-benzotriazole,"MAP2K1, LCK, RPS6KB1, AKT1,PRKCA, MAPK11, MAPK...","PIK3CA,GNB4,DVL3,RFC4,MECOM,CDKN2A, TP53, PIK3...",13,18,72.222222,"inhibitor, inhibitor, inhibitor, inhibitor","ATP binding, 14-3-3 protein binding, [hydroxym...",0.001380,0.004684,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC, SOMATIC","15896313, 16778075, 17974918, 12532415, 145813...",24,"CHEK1, MAPK8, AKT1","17312781, 12702551, 11309301, 16012721, 113588...",82,"DVL3, CDKN2A, TP53, RFC4, PIK3CA"
94,tivozanib,"FGFR1, MET, KDR, PDGFRB, FLT1, KIT, PDGFRA, FL...","PIK3CA, PIK3CA",10,11,90.909091,"inhibitor, inhibitor","ATP binding, ATP binding",0.012987,0.012491,"AMPLIFICATION, SOMATIC","18059337, 15165306, 17684930, 16757203, 183498...",50,"MET, FGFR1, PDGFRA, KIT","16676365, 17848307, 15543611, 15700036, 179891...",34,PIK3CA
97,trilaciclib,"CDK5,CDK2, CDK4, CDK6, CDK7, CDK6, CDK9, CDK5,...","CDKN2A,CDKN2B, TP53, PIK3CA",4,6,66.666667,"inhibitor, inhibitor, inhibitor","acetylcholine receptor activator activity, ATP...",0.028340,0.035446,"DELETION, SOMATIC, SOMATIC","12429646, 12212108, 11585773, 16027075, 164964...",68,"CDK2, CDK4, CDK5, CDK6","17312781, 12702551, 11309301, 17673925, 160127...",64,"PIK3CA, TP53, CDKN2B, CDKN2A"
101,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,0.028340,0.033063,SOMATIC,"16350727, 18059337, 15719371, 17471236, 129076...",55,"RET, FGFR3, FGFR1","16676365, 17848307, 15543611, 15700036, 179891...",17,PIK3CA


In [115]:
### combined direct and indirect final results
### final columns: DRUG   GENE_TARGET CONNECTED_TO (risk gene)	NUM_DIRECT_TARGETS_HIT  
# Number of risk or immediate neighbor genes targeted	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr	
### column for Target description: direct, or indirect
hpv_negative_final_direct['Target_Description'] = 'Direct'
hpv_negative_final_indirect['Target_Description'] = 'Indirect'
hpv_negative_final_results = pd.concat([hpv_negative_final_direct, hpv_negative_final_indirect], ignore_index=True)
### replace any null with 'NA' in the hwole dataframe
hpv_negative_final_results = hpv_negative_final_results.fillna('NA')
hpv_negative_final_results = hpv_negative_final_results.groupby('DRUG').agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'NUM_DIRECT_TARGETS_HIT': 'first',
    'Number of risk or immediate neighbor genes targeted': 'first',
    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'max',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'Target_Description': lambda x: ', '.join(x)
}).reset_index()

In [116]:
len(set(hpv_negative_final_results['DRUG']))

72